In [ ]:
# Cell 1
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
%pip install ipywidgets




Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
!pip install ipywidgets



Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Cell 2
df = pd.read_csv('D:/mini project model/mini project/updated_food_waste_dataset.csv')
print(df.shape)
df.head()


(332, 15)


,Date,Household_Size,Number_of_Meals_Prepared,Income_Level,Food_Type,Shelf_Life,Purchase_Quantity_kg,Consumed_Quantity_kg,Storage_Method,Amount_of_Waste_kg,Reason_for_Waste,Consumption_Ratio,Leftover_Flag,Household_Size_x_Meals,Waste_per_Meal
0,2025-01-03,1,6,High,Dairy,8,9.30,8.06,Room_Temperature,0.47,Not_eaten,0.866667,1,6,0.078333
1,2024-10-14,1,5,Low,Dairy,11,7.22,3.72,Room_Temperature,1.49,Overcooked,0.515235,1,5,0.298000
2,2024-12-29,2,4,Low,Grains,3,6.66,3.56,Refrigerated,1.91,Spoiled,0.534535,1,8,0.477500
3,2024-04-25,3,2,Medium,Dairy,9,4.09,3.02,Refrigerated,1.76,Overcooked,0.738386,1,6,0.880000
4,2024-04-28,2,2,Low,Fruits,1,7.44,5.64,Room_Temperature,0.81,Not_eaten,0.758065,1,4,0.405000


In [ ]:
# Cell 3
# Original numerical and categorical columns (adjust if your names differ)
original_numerical = [
    'Household_Size',
    'Number_of_Meals_Prepared',
    'Shelf_Life',                # if this column still exists
    'Purchase_Quantity_kg',
    'Consumed_Quantity_kg'
]
original_categorical = ['Food_Type', 'Storage_Method', 'Reason_for_Waste']

# Engineered features we added
engineered = [
    'Consumption_Ratio',
    'Leftover_Flag',
    'Household_Size_x_Meals',
    'Waste_per_Meal'
]

# Combine for X, and set y
X = df[ original_numerical + engineered + original_categorical ]
y = df['Amount_of_Waste_kg']

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)



Feature matrix shape: (332, 12)
Target vector shape: (332,)


In [ ]:
# Check for missing values
df.isnull().sum()

Date                        0
Household_Size              0
Number_of_Meals_Prepared    0
Income_Level                0
Food_Type                   0
Shelf_Life                  0
Purchase_Quantity_kg        0
Consumed_Quantity_kg        0
Storage_Method              0
Amount_of_Waste_kg          0
Reason_for_Waste            0
Consumption_Ratio           0
Leftover_Flag               0
Household_Size_x_Meals      0
Waste_per_Meal              0
dtype: int64

In [ ]:
# View data types
df.dtypes

Date                         object
Household_Size                int64
Number_of_Meals_Prepared      int64
Income_Level                 object
Food_Type                    object
Shelf_Life                    int64
Purchase_Quantity_kg        float64
Consumed_Quantity_kg        float64
Storage_Method               object
Amount_of_Waste_kg          float64
Reason_for_Waste             object
Consumption_Ratio           float64
Leftover_Flag                 int64
Household_Size_x_Meals        int64
Waste_per_Meal              float64
dtype: object

In [ ]:
# View dataset statistics
df.describe()

,Household_Size,Number_of_Meals_Prepared,Shelf_Life,Purchase_Quantity_kg,Consumed_Quantity_kg,Amount_of_Waste_kg,Consumption_Ratio,Leftover_Flag,Household_Size_x_Meals,Waste_per_Meal
count,332.000000,332.000000,332.000000,332.000000,332.000000,332.000000,332.000000,332.0,332.000000,332.000000
mean,3.018072,4.978916,6.810241,6.632199,3.406566,1.480422,0.522498,1.0,14.719880,0.469910
std,1.435303,2.584437,3.671168,2.352673,2.143013,0.876484,0.262900,0.0,10.799316,0.559982
min,1.000000,1.000000,1.000000,0.540000,0.110000,0.010000,0.013187,1.0,1.000000,0.003333
25%,2.000000,3.000000,4.000000,5.005000,1.552500,0.647500,0.306064,1.0,6.000000,0.158542
50%,3.000000,5.000000,7.000000,7.080000,3.225000,1.490000,0.539460,1.0,12.000000,0.306333
75%,4.000000,7.000000,10.000000,8.432500,4.970000,2.202500,0.739784,1.0,21.000000,0.516250
max,5.000000,9.000000,13.000000,10.000000,9.000000,2.940000,1.000000,1.0,45.000000,2.910000


In [ ]:
# Distribution of target variable (Amount_of_Waste_kg)
sns.histplot(df['Amount_of_Waste_kg'], bins=20)
plt.title("Distribution of Food Waste (kg)")
plt.show()

NameError: name 'sns' is not defined

In [ ]:
# Separate features (X) and target (y)
X = df.drop('Amount_of_Waste_kg', axis=1)
y = df['Amount_of_Waste_kg']

In [ ]:
# Cell 4
# Specify which columns are numeric vs. categorical
num_cols = original_numerical + engineered
cat_cols = original_categorical

# Create transformers
num_transformer = StandardScaler()
cat_transformer = OneHotEncoder(handle_unknown='ignore')

# ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_transformer, num_cols),
    ('cat', cat_transformer, cat_cols)
])

# Full pipeline with Random Forest
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42))
])


In [ ]:
# List of categorical columns and numerical columns
categorical_columns = ['Food_Type', 'Storage_Method', 'Reason_for_Waste']
numerical_columns = ['Household_Size', 'Number_of_Meals_Prepared', 'Shelf_Life', 'Purchase_Quantity_kg', 'Consumed_Quantity_kg']

In [ ]:
# Cell 5
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model_pipeline.fit(X_train, y_train)


,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
# Cell 6
y_pred = model_pipeline.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print(f"MSE: {mse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R² : {r2:.4f}")


MSE: 0.1417
MAE: 0.2417
R² : 0.8261


In [ ]:
# Cell 7
param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_leaf': [1, 2, 4]
}

grid = GridSearchCV(model_pipeline, param_grid, cv=5, scoring='r2', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print("Best Params:", grid.best_params_)
print("Best CV R²:", grid.best_score_)

# Re-evaluate on test
best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test)
print("Test R² after tuning:", r2_score(y_test, y_pred_best))


Fitting 5 folds for each of 27 candidates, totalling 135 fits
Best Params: {'model__max_depth': None, 'model__min_samples_leaf': 2, 'model__n_estimators': 100}
Best CV R²: 0.9094341173744989
Test R² after tuning: 0.8216968973871528


In [ ]:
# Save pipeline
joblib.dump(model_pipeline, 'food_waste2_pipeline.pkl')

print("Model trained and saved successfully.")

Model trained and saved successfully.


In [ ]:
# Cell 8
sample = {
    'Household_Size': 4,
    'Number_of_Meals_Prepared': 6,
    'Shelf_Life': 3,
    'Purchase_Quantity_kg': 5.0,
    'Consumed_Quantity_kg': 3.5,
    'Food_Type': 'Vegetables',
    'Storage_Method': 'Room_Temperature',
    'Reason_for_Waste': 'Expired',
    # Engineered features you can compute or let the pipeline ignore if missing:
    'Consumption_Ratio': 3.5 / 5.0,
    'Leftover_Flag': 1,
    'Household_Size_x_Meals': 4 * 6,
    'Waste_per_Meal': (5.0-3.5)/6
}
sample_df = pd.DataFrame([sample])
print("Predicted Waste (kg):", best_model.predict(sample_df)[0])


Predicted Waste (kg): 1.4865307142857151


In [ ]:
# Cell 1: Imports & Load Pipeline
import pandas as pd
import joblib

# Load your retrained pipeline (preprocessor + model)
pipeline = joblib.load("D:/mini project model/mini project/food_waste2_pipeline.pkl")

# Cell 2: Prompt the user for inputs
print("Please enter the following details for food waste prediction:")

hs    = float(input("Household Size (number of people): "))
meals = float(input("Number of Meals Prepared: "))
shelf = float(input("Shelf Life (days): "))
buy   = float(input("Purchase Quantity (kg): "))
cons  = float(input("Consumed Quantity (kg): "))
ft    = input("Food Type (e.g. Vegetables, Meat): ")
sm    = input("Storage Method (e.g. Refrigerator): ")
rw    = input("Reason for Waste (e.g. Expired): ")

# Cell 3: Compute engineered features
consumption_ratio = cons / buy if buy else 0
leftover_flag     = 1 if (buy - cons) > 0 else 0
hs_x_meals        = hs * meals
waste_per_meal    = (buy - cons) / meals if meals else 0

# Cell 4: Build a DataFrame for prediction
input_df = pd.DataFrame([{
    'Household_Size':              hs,
    'Number_of_Meals_Prepared':    meals,
    'Shelf_Life':                  shelf,
    'Purchase_Quantity_kg':        buy,
    'Consumed_Quantity_kg':        cons,
    'Consumption_Ratio':           consumption_ratio,
    'Leftover_Flag':               leftover_flag,
    'Household_Size_x_Meals':      hs_x_meals,
    'Waste_per_Meal':              waste_per_meal,
    'Food_Type':                   ft,
    'Storage_Method':              sm,
    'Reason_for_Waste':            rw
}])

# Cell 5: Predict and display result
prediction = pipeline.predict(input_df)[0]
print(f"\n🔮 Predicted Food Waste: {prediction:.2f} kg")


Please enter the following details for food waste prediction:


KeyboardInterrupt: Interrupted by user

In [ ]:
import pandas as pd
import joblib
import ipywidgets as widgets
from IPython.display import display, clear_output

# Load your trained pipeline model
pipeline = joblib.load("D:/mini project model/mini project/food_waste2_pipeline.pkl")

# Create input widgets
hs_widget = widgets.IntSlider(description='Household Size', min=1, max=10, value=4)
meals_widget = widgets.IntSlider(description='Meals Prepared', min=1, max=20, value=6)
shelf_widget = widgets.IntText(description='Shelf Life (days)', value=3)
buy_widget = widgets.FloatText(description='Purchase Qty (kg)', value=5.0)
cons_widget = widgets.FloatText(description='Consumed Qty (kg)', value=3.5)

food_type_widget = widgets.Dropdown(
    description='Food Type',
    options=['Vegetables', 'Fruits', 'Meat', 'Dairy', 'Grains'],
    value='Vegetables'
)

storage_widget = widgets.Dropdown(
    description='Storage Method',
    options=['Refrigerator', 'Freezer', 'Room Temperature'],
    value='Refrigerator'
)

reason_widget = widgets.Dropdown(
    description='Reason for Waste',
    options=['Expired', 'Spoiled', 'Overcooked', 'Leftover'],
    value='Expired'
)

predict_button = widgets.Button(description='Predict Waste')
output = widgets.Output()

# Define what happens when button is clicked
def on_predict_clicked(b):
    with output:
        clear_output()
        hs = hs_widget.value
        meals = meals_widget.value
        shelf = shelf_widget.value
        buy = buy_widget.value
        cons = cons_widget.value
        ft = food_type_widget.value
        sm = storage_widget.value
        rw = reason_widget.value

        # Compute features
        consumption_ratio = cons / buy if buy else 0
        leftover_flag = 1 if (buy - cons) > 0 else 0
        hs_x_meals = hs * meals
        waste_per_meal = (buy - cons) / meals if meals else 0

        # Create dataframe for prediction
        df = pd.DataFrame([{
            'Household_Size': hs,
            'Number_of_Meals_Prepared': meals,
            'Shelf_Life': shelf,
            'Purchase_Quantity_kg': buy,
            'Consumed_Quantity_kg': cons,
            'Consumption_Ratio': consumption_ratio,
            'Leftover_Flag': leftover_flag,
            'Household_Size_x_Meals': hs_x_meals,
            'Waste_per_Meal': waste_per_meal,
            'Food_Type': ft,
            'Storage_Method': sm,
            'Reason_for_Waste': rw
        }])

        # Make prediction
        pred = pipeline.predict(df)[0]
        print(f"🔮 Predicted Food Waste: {pred:.2f} kg")

# Bind the button to function
predict_button.on_click(on_predict_clicked)

# Show all widgets
display(
    hs_widget, meals_widget, shelf_widget, buy_widget, cons_widget,
    food_type_widget, storage_widget, reason_widget,
    predict_button, output
)


IntSlider(value=4, description='Household Size', max=10, min=1)

IntSlider(value=6, description='Meals Prepared', max=20, min=1)

IntText(value=3, description='Shelf Life (days)')

FloatText(value=5.0, description='Purchase Qty (kg)')

FloatText(value=3.5, description='Consumed Qty (kg)')

Dropdown(description='Food Type', options=('Vegetables', 'Fruits', 'Meat', 'Dairy', 'Grains'), value='Vegetabl…

Dropdown(description='Storage Method', options=('Refrigerator', 'Freezer', 'Room Temperature'), value='Refrige…

Dropdown(description='Reason for Waste', options=('Expired', 'Spoiled', 'Overcooked', 'Leftover'), value='Expi…

Button(description='Predict Waste', style=ButtonStyle())

Output()

In [ ]:
import joblib
joblib.dump(model_pipeline, "D:/mini project model/mini project/food_waste2_pipeline.pkl")


['D:/mini project model/mini project/food_waste2_pipeline.pkl']

heloo
